<a href="https://colab.research.google.com/github/mustafablak/cross-domain-turkish-sentiment-analysis/blob/main/notebooks/01_data_preprocessing_and_split_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/Bitirme Projesi/'
train_path = base_path + 'train.csv'
test_path = base_path + 'test.csv'

try:
    print("Loading and concatenating data (This process may take 15-20 seconds)...\n")
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    df_all = pd.concat([df_train, df_test], ignore_index=True)

    # INITIAL MAPPING AND FILTERING
    domain_map = {
        'urun_yorumlari': 'Product Reviews',
        'HUMIR': 'Movie Reviews',
        'tweet-pn': 'Tweets'
    }
    df_filtered = df_all[df_all['dataset'].isin(domain_map.keys())].copy()
    df_filtered['Domain'] = df_filtered['dataset'].map(domain_map)

    # ==========================================
    # SEPARATE HUMIR (Movie vs Hotel)
    # ==========================================
    def separate_humir(text):
        text = str(text).lower()
        hotel_words = ['otel', 'oda', 'personel', 'kahvaltı', 'resepsiyon', 'konaklama', 'tesis', 'tatil', 'temiz']
        movie_words = ['film', 'sinema', 'sahne', 'oyuncu', 'yönetmen', 'senaryo', 'kurgu', 'vizyon', 'karakter']
        if any(word in text for word in hotel_words):
            return 'Hotel Reviews'
        elif any(word in text for word in movie_words):
            return 'Movie Reviews'
        else:
            return 'Unclassified'

    # Apply separation only to the old HUMIR dataset
    mask = df_filtered['Domain'] == 'Movie Reviews'
    df_filtered.loc[mask, 'Domain'] = df_filtered.loc[mask, 'text'].apply(separate_humir)

    # Drop Unclassified texts to maintain high data quality
    df_filtered = df_filtered[df_filtered['Domain'] != 'Unclassified']

    # ==========================================
    # TABLE 2: DATA QUALITY ANALYSIS
    # ==========================================
    missing_data = df_filtered[df_filtered['text'].isnull()].groupby('Domain').size().reset_index(name='Missing Values')
    duplicate_data = df_filtered[df_filtered.duplicated(subset=['text'], keep=False)].groupby('Domain').size().reset_index(name='Duplicate Reviews')

    domains_list = ['Product Reviews', 'Movie Reviews', 'Hotel Reviews', 'Tweets']
    table2 = pd.DataFrame({'Domain': domains_list})
    table2 = pd.merge(table2, missing_data, on='Domain', how='left').fillna(0)
    table2 = pd.merge(table2, duplicate_data, on='Domain', how='left').fillna(0)

    table2['Missing Values'] = table2['Missing Values'].astype(int)
    table2['Duplicate Reviews'] = table2['Duplicate Reviews'].astype(int)

    print("=" * 60)
    print("Table 2 - Data Quality Analysis")
    print("=" * 60)
    print(table2.to_string(index=False))
    print("\n")

    # ==========================================
    # TABLE 3: OVERLAPPING SAMPLES
    # ==========================================
    domain_texts = df_filtered.dropna(subset=['text']).groupby('Domain')['text'].apply(set).to_dict()

    prod_movie = len(domain_texts.get('Product Reviews', set()).intersection(domain_texts.get('Movie Reviews', set())))
    prod_tweet = len(domain_texts.get('Product Reviews', set()).intersection(domain_texts.get('Tweets', set())))
    movie_tweet = len(domain_texts.get('Movie Reviews', set()).intersection(domain_texts.get('Tweets', set())))
    prod_hotel = len(domain_texts.get('Product Reviews', set()).intersection(domain_texts.get('Hotel Reviews', set())))
    movie_hotel = len(domain_texts.get('Movie Reviews', set()).intersection(domain_texts.get('Hotel Reviews', set())))

    table3 = pd.DataFrame({
        'Domain Pair': ['Product-Movie', 'Product-Tweet', 'Movie-Tweet', 'Product-Hotel', 'Movie-Hotel'],
        'Common Samples': [prod_movie, prod_tweet, movie_tweet, prod_hotel, movie_hotel]
    })

    print("=" * 60)
    print("Table 3 - Overlapping Samples")
    print("=" * 60)
    print(table3.to_string(index=False))
    print("\n")

    df_clean = df_filtered.dropna(subset=['text', 'label', 'dataset']).copy()
    df_binary = df_clean[df_clean['label'] != 'Notr'].copy()

    # remove texts that appear with both Positive and Negative labels
    label_counts = df_binary.groupby('text')['label'].nunique()
    conflicting_texts = label_counts[label_counts > 1].index
    print(f"Texts with conflicting labels removed: {len(conflicting_texts)}\n")

    df_binary = df_binary[~df_binary['text'].isin(conflicting_texts)]
    df_binary = df_binary.drop_duplicates(subset=['text']).copy()

    # only Positive / Negative should remain
    assert set(df_binary['label'].unique()) == {'Positive', 'Negative'}

    table1 = df_binary.groupby(['Domain', 'label']).size().unstack(fill_value=0)
    if 'Positive' in table1.columns and 'Negative' in table1.columns:
        table1 = table1[['Positive', 'Negative']]
    table1['Total'] = table1.sum(axis=1)
    table1 = table1.reset_index()

    print("=" * 60)
    print("Table 1 - Domain and Label Distribution")
    print("=" * 60)
    print(table1.to_string(index=False))
    print("\n")

    # ==========================================
    # SPLIT: stratified by Domain + label
    # ==========================================
    print("Saving Natural and Balanced Train/Val/Test Partitions to Drive...")

    # combined key keeps both the domain and the label ratios in every split
    df_binary['strat_key'] = df_binary['Domain'] + '_' + df_binary['label']

    # 70% train, 30% temp
    train_df, temp_df = train_test_split(
        df_binary, test_size=0.30, random_state=42, stratify=df_binary['strat_key'])

    # temp -> 10% val, 20% test of the full data (2/3 of temp goes to test)
    val_df, test_df = train_test_split(
        temp_df, test_size=2/3, random_state=42, stratify=temp_df['strat_key'])

    # check split sizes and label ratios per domain
    total = len(df_binary)
    for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
        print(f"{name}: {len(part)} rows ({len(part) / total:.1%})")
        print(part.groupby('Domain')['label'].value_counts(normalize=True).unstack().round(3))
        print()

    # drop helper column before saving
    cols = [c for c in df_binary.columns if c != 'strat_key']
    train_df = train_df[cols]
    val_df = val_df[cols]
    test_df = test_df[cols]

    train_df.to_csv(base_path + 'train_natural.csv', index=False)
    val_df.to_csv(base_path + 'val_natural.csv', index=False)
    test_df.to_csv(base_path + 'test_natural.csv', index=False)

    def balance_dataset(data):
        balanced_dfs = []
        for domain in data['Domain'].unique():
            domain_data = data[data['Domain'] == domain]
            min_class_size = domain_data['label'].value_counts().min()

            df_pos = domain_data[domain_data['label'] == 'Positive'].sample(n=min_class_size, random_state=42)
            df_neg = domain_data[domain_data['label'] == 'Negative'].sample(n=min_class_size, random_state=42)

            balanced_dfs.append(pd.concat([df_pos, df_neg]))
        return pd.concat(balanced_dfs).sample(frac=1, random_state=42)

    # balancing is applied to the training set only
    train_balanced = balance_dataset(train_df)
    train_balanced.to_csv(base_path + 'train_balanced.csv', index=False)

    print("train_balanced distribution:")
    print(train_balanced.groupby(['Domain', 'label']).size().unstack(fill_value=0))

    print("\n✅ All tables are generated and Train/Val/Test datasets are saved to Drive.")

except Exception as e:
    print(f"An error occurred: {e}")

Mounted at /content/drive
Loading and concatenating data (This process may take 15-20 seconds)...

Table 2 - Data Quality Analysis
         Domain  Missing Values  Duplicate Reviews
Product Reviews               0                149
  Movie Reviews               0                126
  Hotel Reviews               0                454
         Tweets               0                  6


Table 3 - Overlapping Samples
  Domain Pair  Common Samples
Product-Movie               0
Product-Tweet               2
  Movie-Tweet               0
Product-Hotel               0
  Movie-Hotel               0


Texts with conflicting labels removed: 81

Table 1 - Domain and Label Distribution
         Domain  Positive  Negative  Total
  Hotel Reviews      4746      6207  10953
  Movie Reviews     22378     22422  44800
Product Reviews    219252     14782 234034
         Tweets      6084      4966  11050


Saving Natural and Balanced Train/Val/Test Partitions to Drive...
train: 210585 rows (70.0%)
label  